In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

In [2]:
tools = [
    {
        "name": "extract_return_request",
        "description": "Extract a structured return request from a customer message.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": ["string", "null"],
                    "description": "The order ID provided by the customer, or null if missing."
                },
                "item": {
                    "type": ["string", "null"],
                    "description": "The item the customer wants to return, or null if missing."
                },
                "reason": {
                    "type": "string",
                    "enum": [
                        "damaged_item",
                        "wrong_item",
                        "changed_mind",
                        "billing_dispute",
                        "policy_exception",
                        "unclear",
                        "other"
                    ]
                },
                "reason_details": {
                    "type": ["string", "null"],
                    "description": "Additional details when reason is other or unclear."
                },
                "desired_action": {
                    "type": "string",
                    "enum": [
                        "return",
                        "refund",
                        "exchange",
                        "replacement",
                        "unclear"
                    ]
                },
                "evidence_provided": {
                    "type": "boolean",
                    "description": "Whether the customer provided evidence such as a photo or attachment."
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "normal", "high"]
                },
                "missing_information": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    }
                },
                "confidence": {
                    "type": "number",
                    "description": "Confidence score from 0 to 1 for the extraction."
                },
                "human_review_required": {
                    "type": "boolean"
                },
                "human_review_reason": {
                    "type": ["string", "null"]
                }
            },
            "required": [
                "order_id",
                "item",
                "reason",
                "reason_details",
                "desired_action",
                "evidence_provided",
                "urgency",
                "missing_information",
                "confidence",
                "human_review_required",
                "human_review_reason"
            ]
        }
    }
]

In [8]:
customer_message = """The item works perfectly, but it arrived broken, and I need a replacement today"""

message = client.messages.create(
    model=model,
    max_tokens=1000,
    tools=tools,
    tool_choice={
        "type": "tool",
        "name": "extract_return_request"
    },
    messages=[
        {
            "role": "user",
            "content": customer_message
        }
    ]
)
print(message.content[0].model_dump_json(indent=2))

{
  "id": "toolu_014NqggFysCPpva49z6D9HV6",
  "caller": {
    "type": "direct"
  },
  "input": {
    "order_id": "<UNKNOWN>",
    "item": "<UNKNOWN>",
    "reason": "damaged_item",
    "reason_details": "Item arrived broken/damaged. Customer notes it still works perfectly despite physical damage.",
    "desired_action": "replacement",
    "evidence_provided": false,
    "urgency": "high",
    "missing_information": [
      "order_id",
      "item"
    ],
    "confidence": 0.75,
    "human_review_required": true,
    "human_review_reason": "Customer urgently needs a same-day replacement but key details are missing (order ID and item name). Additionally, the customer states the item works perfectly despite arriving broken, which is contradictory and may require clarification or special handling."
  },
  "name": "extract_return_request",
  "type": "tool_use"
}


In [9]:
def validate_return_request(data):
    errors = []

    if data["order_id"] is None:
        errors.append("order_id is missing")

    if data["item"] is None:
        errors.append("item is missing")

    if data["reason"] == "other" and not data["reason_details"]:
        errors.append("reason_details is required when reason is other")

    if data["confidence"] < 0.7:
        errors.append("confidence is below review threshold")

    return errors

In [10]:
def needs_human_review(data):
    if data["confidence"] < 0.7:
        return True

    if data["reason"] in ["unclear", "policy_exception"]:
        return True

    if data["human_review_required"]:
        return True

    return False